<a href="https://colab.research.google.com/github/ZeninKris/zmm-movilidad-predictiva/blob/main/notebooks/03_Feature_Engineering_Clima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import requests

from google.colab import drive
drive.mount('/content/drive')

# 1. Cargamos el esqueleto del Pilar 2
ruta_esqueleto = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos.csv'
df_super = pd.read_csv(ruta_esqueleto)
df_super['fecha_hora'] = pd.to_datetime(df_super['fecha_hora'])

# 2. Descargamos el clima de Open-Meteo (Macroplaza)
lat, lon = 25.6667, -100.3167

# CORRECCIÓN AQUÍ: end_date=2026-03-16
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-01-01&end_date=2026-03-16&hourly=temperature_2m,precipitation&timezone=America%2FMonterrey"

print("Pidiendo datos a la API...")
response = requests.get(url)
data = response.json()

# 3. Construimos el DataFrame del Clima
df_clima = pd.DataFrame({
    'fecha_hora': pd.to_datetime(data['hourly']['time']),
    'temperatura_c': data['hourly']['temperature_2m'],
    'precipitacion_mm': data['hourly']['precipitation']
})

# 4. LA FUSIÓN FINAL DEL CLIMA (Left Merge)
df_super = df_super.merge(df_clima, on='fecha_hora', how='left')

# 5. Guardamos el archivo actualizado en Drive
ruta_salida = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos_clima.csv'
df_super.to_csv(ruta_salida, index=False)

print("\n--- RADIOGRAFÍA DEL CLIMA INYECTADO ---")
print(df_super[df_super['precipitacion_mm'] > 0][['fecha_hora', 'temperatura_c', 'precipitacion_mm']].head(10))
print(f"¡Éxito! Archivo guardado en: {ruta_salida}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pidiendo datos a la API...

--- RADIOGRAFÍA DEL CLIMA INYECTADO ---
             fecha_hora  temperatura_c  precipitacion_mm
482 2023-01-21 02:00:00           13.8               0.2
483 2023-01-21 03:00:00           13.5               0.1
484 2023-01-21 04:00:00           13.2               0.1
485 2023-01-21 05:00:00           12.9               0.1
505 2023-01-22 01:00:00           13.9               0.1
506 2023-01-22 02:00:00           14.4               0.2
541 2023-01-23 13:00:00           16.0               0.1
612 2023-01-26 12:00:00           10.8               0.1
627 2023-01-27 03:00:00            8.1               0.1
628 2023-01-27 04:00:00            6.9               0.1
¡Éxito! Archivo guardado en: /content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos_clima.csv


In [ ]:
import pandas as pd
import numpy as np


# 1. Cargamos el Super DataFrame
ruta_super = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos_clima.csv'
df_super = pd.read_csv(ruta_super)

# 2. Verificación de Idempotencia (Evitamos el KeyError)
if 'precipitacion_mm' in df_super.columns:
    print("Columna 'precipitacion_mm' detectada. Iniciando discretización...")
    cortes = [-1.0, 0.09, 5.0, 20.0, 50.0, 999.0]
    etiquetas = [0, 1, 2, 3, 4]

    # pd.cut hace la magia de clasificar cada hora en su caja correspondiente
    df_super['nivel_lluvia'] = pd.cut(df_super['precipitacion_mm'], bins=cortes, labels=etiquetas)
    df_super['nivel_lluvia'] = df_super['nivel_lluvia'].fillna(0).astype(int)

    # Borramos la columna cruda
    df_super.drop(columns=['precipitacion_mm'], inplace=True)

    # Sobrescribimos el archivo SOLO si hicimos cambios
    df_super.to_csv(ruta_super, index=False)
    print("¡El Clima ha sido domado y guardado!")
else:
    print("La columna 'precipitacion_mm' ya no existe. El archivo ya estaba procesado. No hagas doble trabajo.")

print("\n--- DISTRIBUCIÓN DE HORAS POR NIVEL DE LLUVIA ---")
print(df_super['nivel_lluvia'].value_counts())